# EDA và tối ưu bộ nhớ cho dataset chuyến bay

Notebook này thực hiện:
1. Load CSV trong `data/` với khoảng 2 triệu dòng.
2. Tối ưu memory usage bằng cách tối ưu dtype và dùng `category` khi phù hợp.
3. Hiển thị `shape`, `info`, `describe`, `missing values`.

## 1. Import thư viện cần dùng
Chỉ dùng `pandas` và `numpy` theo yêu cầu.

In [ ]:
import pandas as pd
import numpy as np

## 2. Load dữ liệu với low_memory=False
Giải thích: `low_memory=False` giúp `pandas` phân tích cả cột một lần và tránh cảnh báo chia nhỏ dữ liệu.

In [ ]:
csv_path = '../data/DelayedFlights.csv'

print(f'Loading sample from: {csv_path}')
sample = pd.read_csv(csv_path, nrows=10000, low_memory=False)
sample_memory = sample.memory_usage(deep=True).sum() / 1024**2
print(f'Sample memory usage: {sample_memory:.2f} MB')

sample.head()

Sample memory usage: 2.43 MB


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0,2008,1,3,4,2003.0,1955,2211.0,2225,WN,...,4.0,8.0,0,N,0,NaN,NaN,NaN,NaN,NaN
1,1,2008,1,3,4,754.0,735,1002.0,1000,WN,...,5.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN
2,2,2008,1,3,4,628.0,620,804.0,750,WN,...,3.0,17.0,0,N,0,NaN,NaN,NaN,NaN,NaN
3,4,2008,1,3,4,1829.0,1755,1959.0,1925,WN,...,3.0,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0
4,5,2008,1,3,4,1940.0,1915,2121.0,2110,WN,...,4.0,10.0,0,N,0,NaN,NaN,NaN,NaN,NaN


## 3. Tối ưu dtype và category datatype
- `dtype optimization` nghĩa là gán kiểu dữ liệu nhỏ hơn cho cột số thay vì dùng `int64` hoặc `float64` mặc định.
- `category` là kiểu dữ liệu hiệu quả cho cột chuỗi khi số lượng giá trị duy nhất nhỏ hơn nhiều so với số dòng.

In [6]:
def build_optimized_dtypes(df: pd.DataFrame) -> dict:
    dtypes = {}

    for col in df.select_dtypes(include=['int64']).columns:
        dtypes[col] = pd.to_numeric(df[col], downcast='integer').dtype

    for col in df.select_dtypes(include=['float64']).columns:
        dtypes[col] = pd.to_numeric(df[col], downcast='float').dtype

    for col in df.select_dtypes(include=['object']).columns:
        unique_ratio = df[col].nunique(dropna=False) / len(df)
        if unique_ratio < 0.5:
            dtypes[col] = 'category'

    return dtypes

optimized_dtypes = build_optimized_dtypes(sample)
print('Optimized dtypes sample:')
for col, dtype in optimized_dtypes.items():
    print(f'  {col}: {dtype}')

Optimized dtypes sample:
  Unnamed: 0: int16
  Year: int16
  Month: int8
  DayofMonth: int8
  DayOfWeek: int8
  CRSDepTime: int16
  CRSArrTime: int16
  FlightNum: int16
  Distance: int16
  Cancelled: int8
  Diverted: int8
  DepTime: float32
  ArrTime: float32
  ActualElapsedTime: float32
  CRSElapsedTime: float32
  AirTime: float32
  ArrDelay: float32
  DepDelay: float32
  TaxiIn: float32
  TaxiOut: float32
  CarrierDelay: float32
  WeatherDelay: float32
  NASDelay: float32
  SecurityDelay: float32
  LateAircraftDelay: float32
  UniqueCarrier: category
  TailNum: category
  Origin: category
  Dest: category
  CancellationCode: category


C:\Users\beatb\AppData\Local\Temp\ipykernel_11800\3160547582.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


## 4. Load dữ liệu chính với dtype tối ưu
Bước này sẽ giúp giảm RAM cần dùng khi đọc toàn bộ file.

In [ ]:
print('Đang load toàn bộ dữ liệu bằng chunksize để tránh chờ quá lâu...')
chunk_size = 200_000
chunks = []
for i, chunk in enumerate(pd.read_csv(csv_path, dtype=optimized_dtypes, low_memory=False, chunksize=chunk_size), start=1):
    print(f'  Loaded chunk {i} with {len(chunk)} rows')
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f'DataFrame shape: {df.shape}')
print(f'Total memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

DataFrame shape: (1936758, 30)
Total memory usage: 149.83 MB


## 5. Kiểm tra thông tin cơ bản
Hiển thị `shape`, `info`, `describe`, và missing values.

In [8]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percent': (df.isnull().sum() / len(df)) * 100
})

missing = missing[missing['Missing Count'] > 0] \
            .sort_values(by='Missing Count', ascending=False)

print(missing)

                   Missing Count  Missing Percent
LateAircraftDelay         689270        35.588855
WeatherDelay              689270        35.588855
NASDelay                  689270        35.588855
SecurityDelay             689270        35.588855
CarrierDelay              689270        35.588855
ActualElapsedTime           8387         0.433043
ArrDelay                    8387         0.433043
AirTime                     8387         0.433043
ArrTime                     7110         0.367108
TaxiIn                      7110         0.367108
TaxiOut                      455         0.023493
CRSElapsedTime               198         0.010223
TailNum                        5         0.000258


## 6. Giải thích thêm
- `low_memory=False`: buộc `pandas` đọc toàn bộ cột một lần và suy đoán dtype đúng hơn. Nếu để `True`, `pandas` có thể đọc file theo từng chunk nhỏ và suy đoán dtype khác nhau, dẫn đến cảnh báo và dtype không đồng nhất.
- `dtype optimization`: giảm kích thước bộ nhớ bằng cách dùng `int8`, `int16`, `float32` thay vì kiểu mặc định `int64`/`float64`.
- `category datatype`: dùng cho các cột chuỗi có số lượng giá trị duy nhất nhỏ. `category` lưu giá trị bằng mã số nội bộ và giảm bộ nhớ rất hiệu quả.

## 7. Data Cleaning
Trong phần này ta kiểm tra và xử lý dữ liệu thực tế trước khi phân tích hoặc huấn luyện mô hình.
- Kiểm tra missing values và xử lý hợp lý.
- Loại bỏ duplicated rows để tránh dữ liệu trùng lặp.
- Xử lý outlier cho `ArrDelay` và `DepDelay`.
- Drop các cột không cần thiết hoặc không hữu ích cho bài toán dự đoán.

In [9]:
# 7.1 Kiểm tra missing values
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percent': df.isnull().mean() * 100
})
missing = missing[missing['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
print('Missing values chi tiết theo cột:')
missing

Missing values chi tiết theo cột:


,Missing Count,Missing Percent
LateAircraftDelay,689270,35.588855
WeatherDelay,689270,35.588855
NASDelay,689270,35.588855
SecurityDelay,689270,35.588855
CarrierDelay,689270,35.588855
ActualElapsedTime,8387,0.433043
ArrDelay,8387,0.433043
AirTime,8387,0.433043
ArrTime,7110,0.367108
TaxiIn,7110,0.367108


### Business reasoning cho missing values
- `ArrDelay` và `DepDelay` là mục tiêu chính; nếu thiếu thì không dùng được để huấn luyện hoặc đánh giá độ trễ.
- Cột thời gian, sân bay, hoặc carrier nếu thiếu có thể gây sai lệch phân tích; ta sẽ xử lý theo mức độ quan trọng.

In [10]:
# 7.2 Xử lý missing values hợp lý
critical_cols = [col for col in ['ArrDelay', 'DepDelay'] if col in df.columns]
df = df.dropna(subset=critical_cols)

# Với các cột bổ trợ quan trọng khác, ta có thể impute hoặc loại trừ nếu quá nhiều missing.
for col in ['Origin', 'Dest', 'UniqueCarrier', 'Month', 'DayOfWeek']:
    if col in df.columns and df[col].isnull().any():
        if df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
            df[col] = df[col].fillna(df[col].mode().iloc[0])
        else:
            df[col] = df[col].fillna(df[col].median())

# Những cột số ít missing và có ý nghĩa thấp hơn sẽ được giữ lại nếu cần,
# hoặc có thể bỏ nếu giá trị bị thiếu quá nhiều.
print('Sau khi drop missing ArrDelay/DepDelay:', df.shape)

Sau khi drop missing ArrDelay/DepDelay: (1928371, 30)


### 7.3 Kiểm tra duplicated rows
- Bản ghi trùng lặp có thể khiến mô hình học sai lệch và đánh giá không chính xác.
- Nếu một chuyến bay cùng tất cả trường dữ liệu bị lặp lại, ta chỉ giữ một bản.

In [11]:
before_dup = len(df)
df = df.drop_duplicates()
after_dup = len(df)
print(f'Rows before deduplication: {before_dup}')
print(f'Rows after deduplication: {after_dup}')
print(f'Duplicated rows removed: {before_dup - after_dup}')

Rows before deduplication: 1928371
Rows after deduplication: 1928371
Duplicated rows removed: 0


### 7.4 Xử lý outlier cho ArrDelay và DepDelay
- Flight delay phân bố lệch phải, nên các giá trị lớn vẫn có thể giữ thông tin hoạt động.
- Cần giữ các trường hợp đến/đi sớm hợp lệ (negative delay) và tránh loại bỏ quá nhiều bản ghi đúng.
- Logic dưới đây dùng ngưỡng IQR rộng hơn và ràng buộc thực tế để lọc chỉ những giá trị thực sự bất thường.

In [12]:
def compute_delay_bounds(
    series: pd.Series,
    iqr_multiplier: float = 3.0,
    min_delay: int = -120,
    max_delay: int = 360,
) -> tuple:
    clean = series.dropna()
    q1 = clean.quantile(0.25)
    q3 = clean.quantile(0.75)
    iqr = q3 - q1

    lower_iqr = q1 - iqr_multiplier * iqr
    upper_iqr = q3 + iqr_multiplier * iqr

    lower = max(lower_iqr, min_delay)
    upper = min(upper_iqr, max_delay)

    return lower, upper

outlier_cols = [col for col in ['ArrDelay', 'DepDelay'] if col in df.columns]

bounds = {
    col: compute_delay_bounds(df[col], iqr_multiplier=3.0, min_delay=-120, max_delay=360)
    for col in outlier_cols
}

for col, (lower, upper) in bounds.items():
    print(f'{col}: lower={lower:.2f}, upper={upper:.2f}')

before_outliers = len(df)
mask = pd.Series(True, index=df.index)
for col, (lower, upper) in bounds.items():
    mask &= df[col].between(lower, upper, inclusive='both')

filtered_df = df[mask].copy()
after_outliers = len(filtered_df)
removed_rows = before_outliers - after_outliers
removed_pct = removed_rows / before_outliers * 100

print(f'Rows before filtering: {before_outliers}')
print(f'Rows after filtering: {after_outliers}')
print(f'Removed rows: {removed_rows}')
print(f'Removed percentage: {removed_pct:.2f}%')

df = filtered_df

ArrDelay: lower=-120.00, upper=197.00
DepDelay: lower=-111.00, upper=176.00
Rows before filtering: 1928371
Rows after filtering: 1868454
Removed rows: 59917
Removed percentage: 3.11%


### 7.5 Drop các cột không cần thiết
- Loại bỏ các cột định danh hoặc không liên quan trực tiếp tới dự đoán độ trễ thực tế.
- Ví dụ: `Year` thường không thay đổi nhiều trong bộ dữ liệu, `TailNum` và `FlightNum` là định danh riêng không giúp mô hình tổng quát.
- Cột `Cancelled` / `Diverted` nếu có thể không cần nếu ta chỉ tập trung vào các chuyến bay đã cất cánh.

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# Drop unnecessary columns
# =========================

cols_to_drop = [
    'Year',
    'FlightNum',
    'TailNum',
    'Cancelled',
    'Diverted',
    'CRSDepTime',
    'CRSArrTime',
    'ActualElapsedTime',
    'AirTime',
    'Flights'
]

cols_to_drop = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_to_drop)

print('Dropped columns:')
print(cols_to_drop)

print('\nDataFrame shape after cleanup:')
print(df.shape)


# =========================
# Memory-efficient encoding
# =========================

print('\nMemory usage before encoding:')
print(f'{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

# One-hot encode only low-cardinality columns
low_cardinality_cols = [
    col for col in ['UniqueCarrier']
    if col in df.columns
]

df = pd.get_dummies(
    df,
    columns=low_cardinality_cols,
    prefix=low_cardinality_cols,
    dtype=np.uint8,
    drop_first=True
)

# Convert high-cardinality columns to category
high_cardinality_cols = [
    col for col in ['Origin', 'Dest']
    if col in df.columns
]

for col in high_cardinality_cols:
    df[col] = df[col].astype('category')

# Optional: frequency encoding
for col in high_cardinality_cols:
    freq = df[col].value_counts(normalize=True)
    df[f'{col}_Freq'] = df[col].map(freq).astype(np.float32)

print('\nMemory usage after encoding:')
print(f'{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\nFinal DataFrame shape:')
print(df.shape)

display(df.head())


# Export cleaned dataset
df.to_csv('../data/cleaned_flights.csv', index=False)

print('File exported successfully:')
print('../data/cleaned_flights.csv')

Dropped columns:
[]

DataFrame shape after cleanup:
(1868454, 41)

Memory usage before encoding:
169.30 MB

Memory usage after encoding:
169.30 MB

Final DataFrame shape:
(1868454, 41)


,Unnamed: 0,Month,DayofMonth,DayOfWeek,DepTime,ArrTime,CRSElapsedTime,ArrDelay,DepDelay,Origin,...,UniqueCarrier_NW,UniqueCarrier_OH,UniqueCarrier_OO,UniqueCarrier_UA,UniqueCarrier_US,UniqueCarrier_WN,UniqueCarrier_XE,UniqueCarrier_YV,Origin_Freq,Dest_Freq
0,0,1,3,4,2003.0,2211.0,150.0,-14.0,8.0,IAD,...,0,0,0,0,0,1,0,0,0.011126,0.011779
1,1,1,3,4,754.0,1002.0,145.0,2.0,19.0,IAD,...,0,0,0,0,0,1,0,0,0.011126,0.011779
2,2,1,3,4,628.0,804.0,90.0,14.0,8.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.015411
3,4,1,3,4,1829.0,1959.0,90.0,34.0,34.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.015411
4,5,1,3,4,1940.0,2121.0,115.0,11.0,25.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.004983


File exported successfully:
../data/cleaned_flights.csv


### 7.6 Kết quả Data Cleaning
- Dữ liệu đã được loại bỏ missing giá trị quan trọng, duplicate rows và outlier cực đoan cho `ArrDelay` và `DepDelay`.
- Các cột không cần thiết đã bị drop để giảm nhiễu và giữ lại dữ liệu có giá trị dự đoán.
- Bước này giúp mô hình học tốt hơn và tránh sai lệch do dữ liệu xấu.